# backward-on-scalar-loss — faded example 3: Zero the grad buffer between two backward() calls

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `backward-on-scalar-loss`. The last cell reports your progress on the `PyTorch: backward()` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: backward()` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`backward-on-scalar-loss`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "backward-on-scalar-loss"
DD_SUBTOPIC = "PyTorch: backward()"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`.backward()` accumulates into `.grad`. Without resetting, a second backward doubles the gradient. Calling `w.grad.zero_()` clears the leaf's gradient buffer in place so the next backward yields a fresh, correct gradient.

## Faded exercise 3

### Faded — reset the gradient before the second backward

Implement `faded_zero_grad(w, x, y)`. The first backward runs and its grad is captured. **Complete the missing step: zero the leaf's gradient buffer in place with `w.grad.zero_()`** before the second loss is computed and backpropagated, so the second grad equals the first instead of doubling. Return `(grad_first, grad_second)`.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
def faded_zero_grad(w, x, y):
    loss1 = ((w * x - y) ** 2).mean()
    loss1.backward()
    grad_first = w.grad.clone()
    raise NotImplementedError()  # TODO: fill in this step — read the prompt cell above
    loss2 = ((w * x - y) ** 2).mean()
    loss2.backward()
    grad_second = w.grad.clone()
    return grad_first, grad_second

t.manual_seed(0)
w = t.tensor([0.9], requires_grad=True)
x = t.tensor([1.0, 2.0, 3.0])
y = t.tensor([0.5, 1.0, 2.0])
print(faded_zero_grad(w, x, y))


def _test():
    w = t.tensor([0.9], requires_grad=True)
    x = t.tensor([1.0, 2.0, 3.0])
    y = t.tensor([0.5, 1.0, 2.0])
    g1, g2 = faded_zero_grad(w, x, y)
    assert t.allclose(g1, g2), 'grads must be equal after zero_() reset'
    n = x.numel()
    expected = (2 / n) * ((0.9 * x - y) * x).sum()
    assert t.allclose(g2, expected.reshape(1)), 'second grad must be fresh (not doubled)'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def faded_zero_grad(w, x, y):
    loss1 = ((w * x - y) ** 2).mean()
    loss1.backward()
    grad_first = w.grad.clone()
    w.grad.zero_()
    loss2 = ((w * x - y) ** 2).mean()
    loss2.backward()
    grad_second = w.grad.clone()
    return grad_first, grad_second

t.manual_seed(0)
w = t.tensor([0.9], requires_grad=True)
x = t.tensor([1.0, 2.0, 3.0])
y = t.tensor([0.5, 1.0, 2.0])
print(faded_zero_grad(w, x, y))
```
</details>